# Readout Scheduling: Deterministic Subsets vs Random Queries (Colab)

Notebook 08 extends the modwheel idea to the **readout / inference side**.

Instead of changing the model, we structure which sparse test queries are evaluated first:

```text
many sparse test queries
    -> deterministic subset schedule (mod30-style)
    -> classical model / shadow evaluation
    -> progressive or distributed inference
```

This notebook compares deterministic readout scheduling with random matched query subsets.

Guardrail: this notebook does **not** claim model improvement, QOS improvement, quantum advantage, or a replacement for classical shadows. It evaluates deterministic scheduling as a readout-side workload control pattern.


In [ ]:
# Clone your fork and move into repo root.
# If Colab says the folder already exists, restart runtime or comment these two lines after first run.
!git clone https://github.com/thinkthoughts/quantum-oracle-sketching-mod30.git
%cd quantum-oracle-sketching-mod30


In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from modwheel import STANDARD_WHEELS

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score

fig_dir = Path("figures")
data_dir = Path("data")
fig_dir.mkdir(exist_ok=True)
data_dir.mkdir(exist_ok=True)


## 1. Load 20news and train a baseline classical model

In [ ]:
RANDOM_STATE = 9423
N_RANDOM_TRIALS = 30

categories = [
    "comp.graphics",
    "rec.sport.baseball",
    "sci.space",
    "talk.politics.misc",
]

dataset = fetch_20newsgroups(
    subset="all",
    categories=categories,
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=RANDOM_STATE,
)

texts = np.array(dataset.data, dtype=object)
y = np.array(dataset.target)
target_names = dataset.target_names

texts_train, texts_test, y_train, y_test = train_test_split(
    texts,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

vectorizer = TfidfVectorizer(max_features=12000, min_df=2, stop_words="english")
X_train = vectorizer.fit_transform(texts_train)
X_test = vectorizer.transform(texts_test)

clf = LinearSVC(random_state=RANDOM_STATE, dual="auto")
t0 = time.perf_counter()
clf.fit(X_train, y_train)
train_time = time.perf_counter() - t0

full_pred = clf.predict(X_test)
full_accuracy = accuracy_score(y_test, full_pred)
full_balanced_accuracy = balanced_accuracy_score(y_test, full_pred)

baseline = {
    "n_train": X_train.shape[0],
    "n_test": X_test.shape[0],
    "n_features": X_train.shape[1],
    "train_time_seconds": train_time,
    "full_accuracy": full_accuracy,
    "full_balanced_accuracy": full_balanced_accuracy,
}
baseline


## 2. Define deterministic readout schedules

We treat the test set as many sparse queries. A mod-style schedule selects which queries are evaluated first.

For consistency with earlier notebooks, we test mod30, mod210, and mod2310. The main readout example is mod30.


In [ ]:
def wheel_query_indices(n_queries, wheel_name="mod30"):
    """Return query indices whose index residues are admissible for the wheel."""
    wheel = STANDARD_WHEELS[wheel_name]
    residues = set(wheel.residues)
    return np.array([i for i in range(n_queries) if i % wheel.modulus in residues], dtype=int)


def class_balance_l1_shift(y_subset, y_reference):
    n_classes = len(np.unique(y_reference))
    subset_counts = np.bincount(y_subset, minlength=n_classes)
    ref_counts = np.bincount(y_reference, minlength=n_classes)
    subset_frac = subset_counts / subset_counts.sum()
    ref_frac = ref_counts / ref_counts.sum()
    return float(np.sum(np.abs(subset_frac - ref_frac)))


def evaluate_queries(indices, label, wheel="none", subset_type="deterministic", trial=-1):
    """Evaluate already-trained model on a subset of test queries."""
    t0 = time.perf_counter()
    pred = clf.predict(X_test[indices])
    eval_time = time.perf_counter() - t0
    yt = y_test[indices]
    return {
        "label": label,
        "wheel": wheel,
        "subset_type": subset_type,
        "trial": trial,
        "n_queries_total": X_test.shape[0],
        "n_queries_eval": len(indices),
        "query_fraction": len(indices) / X_test.shape[0],
        "eval_time_seconds": eval_time,
        "accuracy": accuracy_score(yt, pred),
        "balanced_accuracy": balanced_accuracy_score(yt, pred),
        "class_balance_l1_shift": class_balance_l1_shift(yt, y_test),
    }


idx30 = wheel_query_indices(X_test.shape[0], "mod30")
len(idx30), len(idx30) / X_test.shape[0], idx30[:12]


## 3. Matched deterministic vs random readout evaluation

For each wheel, compare deterministic query subset to random query subsets with the same number of evaluated queries.


In [ ]:
results = []

all_idx = np.arange(X_test.shape[0])
results.append(evaluate_queries(all_idx, "full_readout", wheel="none", subset_type="full", trial=-1))

for wheel_name in ["mod30", "mod210", "mod2310"]:
    det_idx = wheel_query_indices(X_test.shape[0], wheel_name)
    n_keep = len(det_idx)
    results.append(evaluate_queries(det_idx, f"{wheel_name}_deterministic", wheel=wheel_name, subset_type="deterministic", trial=-1))

    for trial in range(N_RANDOM_TRIALS):
        rng = np.random.default_rng(RANDOM_STATE + trial * 1009 + len(wheel_name))
        ridx = np.sort(rng.choice(all_idx, size=n_keep, replace=False))
        results.append(evaluate_queries(ridx, f"{wheel_name}_random_{trial:02d}", wheel=wheel_name, subset_type="random_matched", trial=trial))

results_df = pd.DataFrame(results)
results_csv = data_dir / "08_readout_scheduling_results.csv"
results_df.to_csv(results_csv, index=False)
results_df.head(), results_df.tail()


## 4. Summary: deterministic schedule vs random matched mean

In [ ]:
random_summary = (
    results_df[results_df["subset_type"] == "random_matched"]
    .groupby("wheel")
    .agg(
        n_queries_eval=("n_queries_eval", "first"),
        query_fraction=("query_fraction", "first"),
        random_bal_acc_mean=("balanced_accuracy", "mean"),
        random_bal_acc_std=("balanced_accuracy", "std"),
        random_acc_mean=("accuracy", "mean"),
        random_acc_std=("accuracy", "std"),
        random_eval_time_mean=("eval_time_seconds", "mean"),
        random_eval_time_std=("eval_time_seconds", "std"),
        random_class_balance_l1_mean=("class_balance_l1_shift", "mean"),
        random_class_balance_l1_std=("class_balance_l1_shift", "std"),
    )
    .reset_index()
)

deterministic = results_df[results_df["subset_type"] == "deterministic"][[
    "wheel", "n_queries_eval", "query_fraction", "balanced_accuracy", "accuracy", "eval_time_seconds", "class_balance_l1_shift"
]].rename(columns={
    "balanced_accuracy": "det_bal_acc",
    "accuracy": "det_accuracy",
    "eval_time_seconds": "det_eval_time",
    "class_balance_l1_shift": "det_class_balance_l1",
})

summary_df = deterministic.merge(random_summary, on=["wheel", "n_queries_eval", "query_fraction"])
summary_df["delta_bal_acc_vs_random_mean"] = summary_df["det_bal_acc"] - summary_df["random_bal_acc_mean"]
summary_df["delta_acc_vs_random_mean"] = summary_df["det_accuracy"] - summary_df["random_acc_mean"]
summary_df["delta_eval_time_vs_random_mean"] = summary_df["det_eval_time"] - summary_df["random_eval_time_mean"]
summary_df["delta_class_balance_l1_vs_random_mean"] = summary_df["det_class_balance_l1"] - summary_df["random_class_balance_l1_mean"]

summary_csv = data_dir / "08_readout_scheduling_summary.csv"
summary_df.to_csv(summary_csv, index=False)
summary_df


## 5. Progressive readout schedule

In [ ]:
def prefix_indices(indices, fraction):
    k = max(2, int(np.ceil(len(indices) * fraction)))
    return indices[:k]


fractions = np.linspace(0.1, 1.0, 10)
progress_rows = []

for wheel_name in ["mod30", "mod210", "mod2310"]:
    det_all = wheel_query_indices(X_test.shape[0], wheel_name)
    for frac in fractions:
        idx = prefix_indices(det_all, frac)
        row = evaluate_queries(idx, f"{wheel_name}_progress_{frac:.1f}", wheel=wheel_name, subset_type="deterministic_progress", trial=-1)
        row["schedule_fraction"] = frac
        row["fraction_of_all_queries"] = len(idx) / X_test.shape[0]
        progress_rows.append(row)

        for trial in range(10):
            rng = np.random.default_rng(RANDOM_STATE + 50000 + trial * 103 + int(frac * 100) + len(wheel_name))
            ridx = np.sort(rng.choice(all_idx, size=len(idx), replace=False))
            rrow = evaluate_queries(ridx, f"{wheel_name}_random_progress_{frac:.1f}_{trial:02d}", wheel=wheel_name, subset_type="random_progress", trial=trial)
            rrow["schedule_fraction"] = frac
            rrow["fraction_of_all_queries"] = len(ridx) / X_test.shape[0]
            progress_rows.append(rrow)

progress_df = pd.DataFrame(progress_rows)
progress_csv = data_dir / "08_readout_progressive_schedule.csv"
progress_df.to_csv(progress_csv, index=False)
progress_df.head()


## 6. Figure 08a — matched deterministic vs random readout accuracy

In [ ]:
order = ["mod30", "mod210", "mod2310"]
positions = np.arange(1, len(order) + 1)
random_data = [
    results_df[(results_df["wheel"] == w) & (results_df["subset_type"] == "random_matched")]["balanced_accuracy"].values
    for w in order
]
det_points = [
    results_df[(results_df["wheel"] == w) & (results_df["subset_type"] == "deterministic")]["balanced_accuracy"].iloc[0]
    for w in order
]

fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.boxplot(random_data, positions=positions, widths=0.5, labels=order)
ax.scatter(positions, det_points, marker="D", s=70, label="deterministic schedule")
ax.axhline(full_balanced_accuracy, linestyle="--", linewidth=1, label="full readout")
ax.set_title("Readout Scheduling: Deterministic vs Random Matched Queries")
ax.set_xlabel("Schedule / matched query count")
ax.set_ylabel("Balanced accuracy")
ax.set_ylim(0.5, 1.0)
ax.grid(True, axis="y", alpha=0.35)
ax.legend()
fig.tight_layout()
fig_a_svg = fig_dir / "figure_08a_readout_balanced_accuracy_vs_random.svg"
fig_a_png = fig_dir / "figure_08a_readout_balanced_accuracy_vs_random.png"
fig.savefig(fig_a_svg)
fig.savefig(fig_a_png, dpi=220)
plt.show()
fig_a_svg, fig_a_png


## 7. Figure 08b — deterministic delta from random mean

In [ ]:
plot_delta = summary_df.set_index("wheel").loc[order].reset_index()

fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.axhline(0, linestyle="--", linewidth=1)
ax.bar(plot_delta["wheel"], plot_delta["delta_bal_acc_vs_random_mean"])
ax.set_title("Deterministic Readout Accuracy Minus Random Matched Mean")
ax.set_xlabel("Schedule")
ax.set_ylabel("Δ balanced accuracy")
ax.grid(True, axis="y", alpha=0.35)
for i, value in enumerate(plot_delta["delta_bal_acc_vs_random_mean"]):
    ax.text(i, value, f"{value:+.3f}", ha="center", va="bottom" if value >= 0 else "top")
fig.tight_layout()
fig_b_svg = fig_dir / "figure_08b_readout_delta_vs_random.svg"
fig_b_png = fig_dir / "figure_08b_readout_delta_vs_random.png"
fig.savefig(fig_b_svg)
fig.savefig(fig_b_png, dpi=220)
plt.show()
fig_b_svg, fig_b_png


## 8. Figure 08c — progressive readout curve

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 4.8))

for wheel_name in ["mod30", "mod210", "mod2310"]:
    det_prog = progress_df[(progress_df["wheel"] == wheel_name) & (progress_df["subset_type"] == "deterministic_progress")]
    ax.plot(det_prog["fraction_of_all_queries"], det_prog["balanced_accuracy"], marker="o", linewidth=2, label=f"{wheel_name} deterministic")

ax.axhline(full_balanced_accuracy, linestyle="--", linewidth=1, label="full readout")
ax.set_title("Progressive Readout: Accuracy vs Fraction of Queries Evaluated")
ax.set_xlabel("Fraction of all test queries evaluated")
ax.set_ylabel("Balanced accuracy")
ax.set_ylim(0.5, 1.0)
ax.grid(True, alpha=0.35)
ax.legend()
fig.tight_layout()
fig_c_svg = fig_dir / "figure_08c_progressive_readout_curve.svg"
fig_c_png = fig_dir / "figure_08c_progressive_readout_curve.png"
fig.savefig(fig_c_svg)
fig.savefig(fig_c_png, dpi=220)
plt.show()
fig_c_svg, fig_c_png


## 9. Figure 08d — readout evaluation time

In [ ]:
time_rows = results_df[results_df["subset_type"].isin(["full", "deterministic"])].copy()
time_rows["display"] = time_rows["label"].str.replace("_deterministic", "", regex=False).str.replace("full_readout", "full", regex=False)

fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.bar(time_rows["display"], time_rows["eval_time_seconds"])
ax.set_title("Readout Evaluation-Time Proxy")
ax.set_xlabel("Schedule")
ax.set_ylabel("Evaluation time (seconds)")
ax.grid(True, axis="y", alpha=0.35)
fig.tight_layout()
fig_d_svg = fig_dir / "figure_08d_readout_eval_time.svg"
fig_d_png = fig_dir / "figure_08d_readout_eval_time.png"
fig.savefig(fig_d_svg)
fig.savefig(fig_d_png, dpi=220)
plt.show()
fig_d_svg, fig_d_png


## 10. Interpretation helper

In [ ]:
for _, row in summary_df.iterrows():
    print(
        f"{row['wheel']}: query_fraction={row['query_fraction']:.3f}, "
        f"det_bal_acc={row['det_bal_acc']:.3f}, "
        f"random_mean={row['random_bal_acc_mean']:.3f}, "
        f"delta={row['delta_bal_acc_vs_random_mean']:+.3f}"
    )

print("""
Notebook 08 interpretation:

- Deterministic mod-style schedules can structure evaluation of many sparse test queries.
- This does not change the model; it controls how readout/inference work is distributed.
- Compare deterministic schedules to random matched subsets to avoid overclaiming.
- If deterministic ≈ random, the value is reproducibility and scheduling structure, not accuracy gain.
- The progressive curve shows staged inference: evaluate a subset first, then expand if needed.

Guardrail:
This is a readout-side scheduling experiment, not a proof of QOS improvement or quantum advantage.
""")


## 11. Download outputs

In [ ]:
from google.colab import files

for path in [
    "data/08_readout_scheduling_results.csv",
    "data/08_readout_scheduling_summary.csv",
    "data/08_readout_progressive_schedule.csv",
    "figures/figure_08a_readout_balanced_accuracy_vs_random.svg",
    "figures/figure_08b_readout_delta_vs_random.svg",
    "figures/figure_08c_progressive_readout_curve.svg",
    "figures/figure_08d_readout_eval_time.svg",
]:
    files.download(path)
